In [1]:
!pip install finrl

In [2]:
!pip install stable-baselines3

In [3]:
!pip install yfinance

In [6]:
!pip install alpaca-trade-api
!pip install exchange_calendars

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 198.0/198.0 kB 9.0 MB/s eta 0:00:00


In [11]:
!pip install stockstats
!pip install wrds

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 50.0 MB/s eta 0:00:00


In [29]:
import pandas as pd
import numpy as np
from finrl.meta.env_stock_trading.env_stocktrading_np import StockTradingEnv
from finrl.agents.stablebaselines3.models import A2C
from finrl.agents.stablebaselines3.models import PPO
from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
from finrl.meta.preprocessor.preprocessors import FeatureEngineer
from stable_baselines3 import A2C as SB_A2C, PPO as SB_PPO

In [67]:
# Step 1: Load & Preprocess TSX Data
stock_data = pd.read_excel("/content/TSX Chart Data.xlsx")

In [69]:
# Convert date column to datetime
stock_data['date'] = pd.to_datetime(stock_data['date'])

In [70]:
# Sort by date and ticker
stock_data = stock_data.sort_values(by=['date', 'tic']).reset_index(drop=True)

In [25]:
# Step 2: Feature Engineering
# fe = FeatureEngineer(use_technical_indicator=True)
# stock_data = fe.preprocess_data(stock_data)

Successfully added technical indicators


In [71]:
stock_data

,date,close,high,low,open,volume,tic,day
0,2004-01-01,0.105000,0.10500,0.105000,0.10500,0,DYA.TO,3
1,2004-01-01,1.378000,2.65000,2.650000,2.65000,0,EDT.TO,3
2,2004-01-02,0.489625,0.53000,0.530000,0.53000,0,AAB.TO,4
3,2004-01-02,22.070272,29.85000,29.320000,29.43000,413000,ABX.TO,4
4,2004-01-02,3.484277,7.05000,7.050000,7.05000,100,ACD.TO,4
...,...,...,...,...,...,...,...,...
1048570,2024-12-27,21.600000,21.65000,21.219999,21.40000,153600,ELD.TO,4
1048571,2024-12-27,0.370000,0.38000,0.360000,0.36000,16900,ELEF.TO,4
1048572,2024-12-27,1311.250000,1319.98999,1299.520020,1299.52002,1200,ELF.TO,4
1048573,2024-12-27,0.850000,0.88000,0.840000,0.88000,116800,ELO.TO,4


In [72]:
# Step 3: Define A2C Model for Stock Selection
train_data = stock_data.copy()

In [73]:
# Define the list of technical indicators (you can customize this)
tech_indicator_list = ['macd', 'rsi_30', 'cci_30', 'dx_30']

# Count unique stocks in the dataset
stock_dim = len(stock_data["tic"].unique())

In [74]:
stock_data.head(5)

,date,close,high,low,open,volume,tic,day
0,2004-01-01,0.105000,0.105,0.105,0.105,0,DYA.TO,3
1,2004-01-01,1.378000,2.650,2.650,2.650,0,EDT.TO,3
2,2004-01-02,0.489625,0.530,0.530,0.530,0,AAB.TO,4
3,2004-01-02,22.070272,29.850,29.320,29.430,413000,ABX.TO,4
4,2004-01-02,3.484277,7.050,7.050,7.050,100,ACD.TO,4


In [75]:
'close' in stock_data.columns

True

In [81]:
stock_data["date"] = pd.to_datetime(stock_data["date"])
stock_data = stock_data.sort_values(by=["date", "tic"]).reset_index(drop=True)
stock_dim = len(stock_data["tic"].unique())
num_stock_shares = [0] * stock_dim
tech_indicator_list = ["macd", "rsi_30", "cci_30", "dx_30"]
state_space = 1 + stock_dim * (len(tech_indicator_list) + 2)
action_space = stock_dim

from finrl.meta.env_stock_trading.env_stocktrading_np import StockTradingEnv

ppo_env = StockTradingEnv(
    initial_stocks=stock_data,
    config={
      'hmax': 100,
      'stock_dim': stock_dim,
      'initial_amount': 100000,
      'buy_cost_pct': 0.001,
      'sell_cost_pct': 0.001,
      'reward_scaling': 1e-4,
      'tech_indicator_list': tech_indicator_list,
      'num_stock_shares': num_stock_shares,
      'state_space': state_space,
      'action_space': action_space
    }
)

# Print for debugging
print("Stock Dimension:", stock_dim)
print("State Space:", state_space)
print("Action Space:", action_space)


KeyError: 'price_array'

In [60]:
print(stock_data.head())  # Display first few rows
print(stock_data.dtypes)  # Check data types of all columns

           date   close   high    low   open  volume     tic  day     macd  \
0    2004-01-01  0.1050  0.105  0.105  0.105       0  DYA.TO    3  0.00000   
5280 2004-01-01  1.3780  2.650  2.650  2.650       0  EDT.TO    3  0.00000   
1    2004-01-02  0.1050  0.105  0.105  0.105       0  DYA.TO    4  0.00000   
5281 2004-01-02  1.4248  2.740  2.740  2.740     200  EDT.TO    4  0.00105   
2    2004-01-05  0.1050  0.105  0.105  0.105       0  DYA.TO    0  0.00000   

       boll_ub   boll_lb  rsi_30     cci_30  dx_30  close_30_sma  close_60_sma  
0     0.105000  0.105000   100.0  66.666667  100.0        0.1050        0.1050  
5280  0.105000  0.105000   100.0  66.666667  100.0        1.3780        1.3780  
1     0.105000  0.105000   100.0  66.666667  100.0        0.1050        0.1050  
5281  1.467585  1.335215   100.0  66.666667  100.0        1.4014        1.4014  
2     0.105000  0.105000   100.0  66.666667  100.0        0.1050        0.1050  
date            datetime64[ns]
close         